# Paying for APIs with x402 — an Agents SDK example

This notebook shows an OpenAI Agents SDK agent calling **paid HTTP APIs without an API key**. The agent has three tools: a sanctions screen on any wallet address, a SHA-256 helper, and a dual-chain on-chain anchor. Two of those tools cost USDC on Base mainnet; the agent auto-pays from a wallet you control using the [x402 payment protocol](https://x402.org).

**Why this is interesting:** the Agents SDK is great at deciding *when* to call a tool, but every paid tool normally needs its own API-key wiring (Stripe, Alchemy, Twilio, etc.). x402 replaces all of that with a single Base wallet — any HTTP endpoint that wants payment returns `402 Payment Required` with the price, the client signs a small USDC authorization in-memory, retries, and the agent gets the response back. From the agent's perspective every paid tool is just an `await http.get(...)`.

**What you will build:**

1. An x402 httpx client wired to a Base mainnet wallet
2. Three `@function_tool` functions — sanctions screen, sha256, dual-chain anchor
3. A small Compliance Agent that vets wallet addresses

**Cost per run:** ~$0.006 USDC (one screen + one anchor) plus a few hundred input/output tokens of `gpt-5.4-mini`. The on-chain anchor produces real Base + Solana transactions that anyone can verify in a block explorer.

## What the agent does

```
you: "Vet this wallet: 0xd8dA…6045"
        │
        ▼
  tool: screen_wallet("0xd8dA…6045")    ($0.001 USDC)
        GET https://api.anchor-x402.com/v1/screen?wallet=…
        → 402 + { price, payTo, network }
        → client signs EIP-3009 transferWithAuthorization, retries
        → 200 + { sanctions_match, risk_level, … }
        │
        ▼
  tool: sha256_of(verdict_json)          (free, local)
        │
        ▼
  tool: anchor_hash(hash, note=…)        ($0.005 USDC)
        POST https://api.anchor-x402.com/v1/anchor
        → 402 → sign → retry
        → 200 + { base.tx, solana.tx, explorer_urls }
        │
        ▼
agent: "Verdict: clean. Anchored on Base + Solana. URLs: …"
```

## Setup

Install two packages:

- [`openai-agents`](https://github.com/openai/openai-agents-python) — the Agents SDK
- [`x402[httpx,evm]`](https://pypi.org/project/x402/) — official Python client for the x402 protocol. The `httpx` extra adds the `httpx`-based HTTP wrapper; the `evm` extra pulls in `eth-account` + `web3`, required by the Base signer.

In [ ]:
%pip install --quiet "openai-agents>=0.17.0" "x402[httpx,evm]>=2.10.0"

Set two environment variables before running:

- `OPENAI_API_KEY` — your normal OpenAI API key
- `BASE_PRIVATE_KEY` — a 0x-prefixed private key for a **fresh agent-only EOA**, never your main wallet. Fund it with ~$1 USDC on Base; the Coinbase CDP facilitator fronts gas, so you don't need ETH on it.

In a notebook you would normally use `getpass` or a `.env` loader. Here we assume both are already set in the kernel's environment.

In [2]:
import os, sys

assert os.environ.get("OPENAI_API_KEY"),    "set OPENAI_API_KEY before running"
assert os.environ.get("BASE_PRIVATE_KEY"),  "set BASE_PRIVATE_KEY before running"
print("env: OK")

env: OK


## Build the x402 client

This is the only x402-specific scaffolding in the notebook. We construct an `eth_account.LocalAccount` from the private key, wrap it as an `EthAccountSigner`, register the `ExactEvmScheme` on Base mainnet, and pass the resulting client to `x402HttpxClient` whenever a tool needs to make a paid request.

A single `x402Client` instance can serve any number of tools — the per-request `x402HttpxClient` context manager just wraps an `httpx.AsyncClient` with an x402-aware transport.

In [3]:
from eth_account import Account
from x402.client import x402Client
from x402.http.clients.httpx import x402HttpxClient
from x402.mechanisms.evm.exact.client import ExactEvmScheme
from x402.mechanisms.evm.signers import EthAccountSigner

ANCHOR_X402 = "https://api.anchor-x402.com"
BASE_MAINNET = "eip155:8453"

account  = Account.from_key(os.environ["BASE_PRIVATE_KEY"])
signer   = EthAccountSigner(account)
x402     = x402Client().register(BASE_MAINNET, ExactEvmScheme(signer))

print(f"buyer EOA: {account.address}")
print(f"registered: {x402.get_registered_schemes()}")

buyer EOA: 0xb8e2E8d1b2f2Cb91aA603acd77eDCA33C8D39F7C
registered: {1: [], 2: [{'network': 'eip155:8453', 'scheme': 'exact'}]}


## Define the agent's tools

The Agents SDK turns plain async Python functions into tools the model can call via `@function_tool`. Each tool's docstring becomes the description shown to the model — so the descriptions should explain *what the tool does, how much it costs, and what it returns* (the model uses cost to decide whether to call it).

We define three tools:

In [4]:
import hashlib
from agents import Agent, Runner, function_tool


@function_tool
async def screen_wallet(wallet: str) -> dict:
    """Sanctions + AML screen for a Base or Ethereum wallet address.

    Costs ~$0.001 USDC, auto-paid via the x402 protocol. Returns
    `sanctions_match`, `sanctioned_lists`, and a `risk_level` tier.
    """
    async with x402HttpxClient(x402) as http:
        r = await http.get(f"{ANCHOR_X402}/v1/screen?wallet={wallet}")
        r.raise_for_status()
        return r.json()


@function_tool
async def anchor_hash(hex_hash: str, note: str | None = None) -> dict:
    """Dual-chain anchor a 32-byte hex hash on Base + Solana mainnet.

    Costs ~$0.005 USDC, auto-paid via the x402 protocol. Returns the
    resulting `base.tx`, `solana.tx`, and explorer URLs — anyone can
    later re-compute the same hash client-side and verify it matches.
    """
    body = {"hash": hex_hash, "note": note} if note else {"hash": hex_hash}
    async with x402HttpxClient(x402) as http:
        r = await http.post(f"{ANCHOR_X402}/v1/anchor", json=body)
        r.raise_for_status()
        return r.json()


@function_tool
def sha256_of(data: str) -> str:
    """Compute SHA-256 of a UTF-8 string. Free, client-side."""
    return hashlib.sha256(data.encode("utf-8")).hexdigest()

## Define the agent

A short `instructions` block tells the model what order to call the tools in. The model still chooses when (and whether) to call each one based on the user's request — we're just giving it the canonical flow.

In [ ]:
compliance_agent = Agent(
    name="Compliance Agent",
    model="gpt-5.4-mini",
    instructions=(
        "You vet wallet addresses for sanctions risk. For each address the user "
        "gives you: (1) call `screen_wallet` to get a sanctions verdict; "
        "(2) compute a SHA-256 of the verdict JSON via `sha256_of`; "
        "(3) call `anchor_hash` with that hash + a short `note` so the "
        "verdict is dual-chain anchored on Base + Solana for later audit. "
        "Report the verdict in plain English plus the two on-chain explorer "
        "URLs so the user can verify independently."
    ),
    tools=[screen_wallet, sha256_of, anchor_hash],
)

## Run it end-to-end

We give the agent a known wallet to vet — Vitalik's primary address. The cell below makes one OpenAI call, three tool calls, two x402-paid HTTP calls, and writes two real on-chain transactions. Total spend: ~$0.006 USDC + ~$0.001 in OpenAI tokens, in about 4 seconds.

In [6]:
import asyncio

result = await Runner.run(
    compliance_agent,
    "Vet this wallet: 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045",
)
print(result.final_output)

Verdict: no sanctions match; risk level low.

Anchored audit:
- Base: https://basescan.org/tx/0x5f9281a08b859d2331bac9fb7d52afb1fc4697454753c8198cba6b319f3fe49b
- Solana: https://solscan.io/tx/5qMdsXigZPb57XRBkhEuQ8nfhLy5HAaF68rctHthAUeavbTYLUfnc8BJXxKPjch6cCD4Ca5cxgaAEsmjSFsBzZWj

Hash: 4386bdda10563073b7db0c6f35a61bb6d403968dc8822d08bba87cc895c79ac7


The two explorer URLs in the agent's output above are real and clickable — both transactions actually settled on-chain. Anyone scanning Basescan or Solscan can verify the hash matches what the agent reported, which is the whole point of dual-chain anchoring.

## What just happened

- **Discovery.** The screen tool hit `GET /v1/screen?wallet=…`. The server returned `402 Payment Required` with a JSON body listing accepted prices (USDC on Base + USDC on Solana) and recipient addresses.
- **Sign.** `x402HttpxClient`'s transport caught the 402, picked the Base option, asked our `EthAccountSigner` to sign an EIP-3009 `transferWithAuthorization` for exactly $0.001 USDC to the seller, attached the signed payload as an `X-Payment` header, and replayed the request.
- **Settle.** The server forwarded the signed payload to the [Coinbase CDP x402 facilitator](https://docs.cdp.coinbase.com/x402), which submitted the on-chain settlement. The facilitator fronts the gas.
- **Respond.** Once settlement succeeded, the server ran the actual screen and returned the verdict as a normal `200 OK` JSON response.

From the agent's perspective, none of that was visible. It just called a tool and got a result.

The anchor tool followed the same pattern, plus a write-side effect: the server hashed the request into a Merkle root, wrote it to Base mainnet calldata and the Solana Memo program in parallel, and returned both transaction signatures.

## Cost recap

| Step | Cost |
|---|---|
| OpenAI model (gpt-5.4-mini, ~600 tokens in + ~150 out) | ~$0.001 |
| `screen_wallet` x402 call | $0.001 USDC |
| `sha256_of` (local) | $0 |
| `anchor_hash` x402 call | $0.005 USDC |
| **Total per run** | **~$0.006 USDC + ~$0.001 OpenAI** |

Note that gas is paid by the Coinbase CDP facilitator, not by the agent's EOA — you only need USDC in the wallet, not ETH.

## Where to go from here

The seller in this notebook is [anchor-x402.com](https://anchor-x402.com), a community-run x402 v2 service catalog. It exposes 16 paid endpoints on Base + Solana — sanctions screening, wallet intel, hash anchoring, signature attestation, tx/calldata decoding, ENS/SNS resolution, USD price, datetime parsing, a $7.77 agent-driven due-diligence investigator, a verifiable RNG, plus five universal LLM endpoints (roast, oracle, tldr, aura, grade). The full catalog is at [`/.well-known/x402.json`](https://anchor-x402.com/.well-known/x402.json) and there are short demo videos at [anchor-x402.com/demos/](https://anchor-x402.com/demos/).

Other things to try with the same agent + wallet setup:

- **Different agent persona** — swap the Compliance Agent for a research agent that calls `/v1/intel/wallet` ($0.005, returns balances + activity + identity in one call) instead of screening.
- **Multi-step due diligence** — wire `/v1/investigate` ($7.77, async, returns a signed markdown report). Treat it as a single big tool the agent calls when stakes are high.
- **Bring your own seller** — any x402-compliant endpoint works. The `x402Client.register()` call accepts multiple network/scheme pairs; just register your seller's network and the same agent can pay both.
- **Move from `private_key_to_account` to a smart wallet** — the same `EthAccountSigner` interface plugs into Coinbase Smart Wallet signers, hardware wallets, or any other [ERC-1271](https://eips.ethereum.org/EIPS/eip-1271)-compatible signing surface.

The x402 spec is at [x402.org](https://x402.org). The Python client is open source: [github.com/coinbase/x402](https://github.com/coinbase/x402).